# 下载 CODa 数据集

递归下载 CODa 目录中的全部文件，支持并行下载、自动重试、跳过已完成文件和 `.part` 断点续传。

> 数据量达到数 TB。请先修改并运行配置单元，再运行扫描单元；确认文件数量后，最后运行下载单元。

## 1. 连接 Google Drive

运行下一单元后，按 Colab 提示选择 Google 账号并授权。Notebook 将在 `MyDrive/datasets` 下创建数据集目录。若使用共享云端硬盘，可在配置单元中改为 `/content/drive/Shareddrives/<共享盘名称>/datasets`。

In [1]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT_POINT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)

GOOGLE_DRIVE_DATASETS = DRIVE_MOUNT_POINT / "MyDrive" / "datasets"
GOOGLE_DRIVE_DATASETS.mkdir(parents=True, exist_ok=True)
print(f"Google Drive 数据集根目录: {GOOGLE_DRIVE_DATASETS}")

KeyboardInterrupt: 

## 2. 安装进度条组件并配置下载

Colab 通常已包含 `tqdm`，下面的安装单元用于确保组件可用。

In [ ]:
%pip install -q "tqdm>=4.66"

In [ ]:
from pathlib import Path

BASE_URL = "https://web.corral.tacc.utexas.edu/texasrobotics/web_CODa/"
DESTINATION = GOOGLE_DRIVE_DATASETS / "CODa"
WORKERS = 3
RETRIES = 5
TIMEOUT = 60
CHUNK_SIZE = 8 * 1024 * 1024

print(f"下载目录: {DESTINATION}")
print(f"并行下载数: {WORKERS}")

In [ ]:
import concurrent.futures
import os
import time
from collections import deque
from dataclasses import dataclass
from html.parser import HTMLParser
from pathlib import PurePosixPath
from urllib.error import HTTPError, URLError
from urllib.parse import unquote, urljoin, urlsplit, urlunsplit
from urllib.request import Request, urlopen
from tqdm.auto import tqdm

USER_AGENT = "CODa-dataset-notebook/1.0"


class LinkParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.casefold() != "a":
            return
        for name, value in attrs:
            if name.casefold() == "href" and value:
                self.links.append(value)
                break


@dataclass(frozen=True)
class Download:
    url: str
    relative_path: Path


def normalized_root_url(value):
    parsed = urlsplit(value)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("BASE_URL 必须是有效的 HTTP(S) URL")
    path = parsed.path if parsed.path.endswith("/") else parsed.path + "/"
    return urlunsplit((parsed.scheme, parsed.netloc, path, "", ""))


def relative_path_for_url(url, root_url):
    parsed, root = urlsplit(url), urlsplit(root_url)
    if parsed.scheme != root.scheme or parsed.netloc != root.netloc:
        return None
    if not parsed.path.startswith(root.path):
        return None
    encoded_relative = parsed.path[len(root.path):]
    if not encoded_relative:
        return Path()
    parts = []
    for encoded_part in PurePosixPath(encoded_relative).parts:
        part = unquote(encoded_part)
        if part in {"", ".", ".."} or "/" in part or "\\" in part or "\x00" in part:
            return None
        parts.append(part)
    return Path(*parts)


def retry(operation, description):
    last_error = None
    for attempt in range(1, RETRIES + 1):
        try:
            return operation()
        except (HTTPError, URLError, TimeoutError, OSError) as error:
            last_error = error
            if attempt == RETRIES:
                break
            delay = min(2 ** (attempt - 1), 30)
            print(f"{description}失败（{attempt}/{RETRIES}）：{error}；{delay} 秒后重试")
            time.sleep(delay)
    raise last_error


def read_url(url):
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=TIMEOUT) as response:
        return response.read()


def discover_files(root_url, destination):
    root_url = normalized_root_url(root_url)
    pending = deque([root_url])
    visited = set()
    files = {}
    while pending:
        directory_url = pending.popleft()
        if directory_url in visited:
            continue
        visited.add(directory_url)
        relative_directory = relative_path_for_url(directory_url, root_url)
        print(f"扫描: {relative_directory or Path('.')}")
        page = retry(lambda: read_url(directory_url), f"读取 {directory_url}")
        parser = LinkParser()
        parser.feed(page.decode("utf-8", errors="replace"))
        for href in parser.links:
            joined = urljoin(directory_url, href)
            parsed = urlsplit(joined)
            clean_url = urlunsplit((parsed.scheme, parsed.netloc, parsed.path, "", ""))
            relative_path = relative_path_for_url(clean_url, root_url)
            if relative_path is None or not relative_path.parts:
                continue
            if parsed.path.endswith("/"):
                pending.append(clean_url)
                (destination / relative_path).mkdir(parents=True, exist_ok=True)
            else:
                files.setdefault(clean_url, Download(clean_url, relative_path))
    return sorted(files.values(), key=lambda item: str(item.relative_path))


def download_once(item, destination):
    target = destination / item.relative_path
    partial = target.with_name(target.name + ".part")
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file():
        return "跳过"
    if target.exists():
        raise OSError(f"目标存在但不是普通文件: {target}")

    existing_size = partial.stat().st_size if partial.is_file() else 0
    headers = {"User-Agent": USER_AGENT}
    if existing_size:
        headers["Range"] = f"bytes={existing_size}-"
    try:
        response = urlopen(Request(item.url, headers=headers), timeout=TIMEOUT)
    except HTTPError as error:
        if error.code == 416 and existing_size:
            if error.headers.get("Content-Range", "") == f"bytes */{existing_size}":
                os.replace(partial, target)
                return "完成"
        raise

    with response:
        status = getattr(response, "status", response.getcode())
        append = existing_size > 0 and status == 206
        if append and not response.headers.get("Content-Range", "").startswith(f"bytes {existing_size}-"):
            raise OSError("服务器返回了不匹配的 Content-Range")
        mode = "ab" if append else "wb"
        expected = response.headers.get("Content-Length")
        expected_bytes = int(expected) if expected and expected.isdigit() else None
        initial_bytes = existing_size if append else 0
        total_bytes = initial_bytes + expected_bytes if expected_bytes is not None else None
        received = 0
        with partial.open(mode) as output, tqdm(
            total=total_bytes,
            initial=initial_bytes,
            desc=str(item.relative_path),
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            mininterval=0.5,
            dynamic_ncols=True,
            leave=True,
        ) as progress_bar:
            while True:
                chunk = response.read(CHUNK_SIZE)
                if not chunk:
                    break
                output.write(chunk)
                received += len(chunk)
                progress_bar.update(len(chunk))
        if expected_bytes is not None and received != expected_bytes:
            raise OSError(f"响应不完整：预期 {expected_bytes}，实际 {received} 字节")
    os.replace(partial, target)
    return "完成"


def download_with_retries(item, destination):
    try:
        result = retry(lambda: download_once(item, destination), f"下载 {item.relative_path}")
        return item, result, None
    except Exception as error:
        return item, "失败", error

## 3. 扫描文件

此单元只读取目录索引，不下载数据文件。

In [3]:
DESTINATION.mkdir(parents=True, exist_ok=True)
files = discover_files(BASE_URL, DESTINATION)
print(f"扫描完成，共发现 {len(files)} 个文件。")
files[:10]

扫描: .
扫描: CODa_models
扫描: CODa_v1
扫描: CODa_vslam
扫描: depthonly
扫描: pretrained_models
扫描: sequences
扫描: splits
扫描: CODa_v1/sequences
扫描: CODa_v1/splits
扫描: pretrained_models/128channel
扫描: pretrained_models/16channel
扫描: pretrained_models/32channel
扫描: pretrained_models/64channel
扫描: splits/CODa_tiny
扫描: CODa_v1/splits/CODa_tiny_split
扫描: splits/CODa_tiny/2d_rect
扫描: splits/CODa_tiny/3d_bbox
扫描: splits/CODa_tiny/3d_comp
扫描: splits/CODa_tiny/3d_raw
扫描: splits/CODa_tiny/3d_semantic
扫描: splits/CODa_tiny/calibrations
扫描: splits/CODa_tiny/metadata
扫描: splits/CODa_tiny/poses
扫描: splits/CODa_tiny/timestamps
扫描: splits/CODa_tiny/2d_rect/cam0
扫描: splits/CODa_tiny/2d_rect/cam1
扫描: splits/CODa_tiny/3d_bbox/os1
扫描: splits/CODa_tiny/3d_comp/os1
扫描: splits/CODa_tiny/3d_raw/cam3
扫描: splits/CODa_tiny/3d_semantic/os1
扫描: splits/CODa_tiny/calibrations/0
扫描: splits/CODa_tiny/calibrations/1
扫描: splits/CODa_tiny/calibrations/10
扫描: splits/CODa_tiny/calibrations/11
扫描: splits/CODa_tiny/calibrations/12
扫描: sp

[Download(url='https://web.corral.tacc.utexas.edu/texasrobotics/web_CODa/CODa_models/2d_bbox.zip', relative_path=PosixPath('CODa_models/2d_bbox.zip')),
 Download(url='https://web.corral.tacc.utexas.edu/texasrobotics/web_CODa/CODa_v1/sequences/0.zip', relative_path=PosixPath('CODa_v1/sequences/0.zip')),
 Download(url='https://web.corral.tacc.utexas.edu/texasrobotics/web_CODa/CODa_v1/sequences/1.zip', relative_path=PosixPath('CODa_v1/sequences/1.zip')),
 Download(url='https://web.corral.tacc.utexas.edu/texasrobotics/web_CODa/CODa_v1/sequences/10.zip', relative_path=PosixPath('CODa_v1/sequences/10.zip')),
 Download(url='https://web.corral.tacc.utexas.edu/texasrobotics/web_CODa/CODa_v1/sequences/11.zip', relative_path=PosixPath('CODa_v1/sequences/11.zip')),
 Download(url='https://web.corral.tacc.utexas.edu/texasrobotics/web_CODa/CODa_v1/sequences/12.zip', relative_path=PosixPath('CODa_v1/sequences/12.zip')),
 Download(url='https://web.corral.tacc.utexas.edu/texasrobotics/web_CODa/CODa_v1/s

## 4. 开始下载

运行以下单元将正式开始多 TB 下载。每个正在下载的文件都有独立进度条，显示已下载量、总大小、速度和预计剩余时间。中断后重新运行此单元即可续传 `.part` 文件。

In [ ]:
failures = []
completed = skipped = 0

with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = [executor.submit(download_with_retries, item, DESTINATION) for item in files]
    for index, future in enumerate(concurrent.futures.as_completed(futures), start=1):
        item, result, error = future.result()
        if result == "完成":
            completed += 1
        elif result == "跳过":
            skipped += 1
        else:
            failures.append((item, error))
        print(f"[{index}/{len(files)}] {result}: {item.relative_path}" + (f" — {error}" if error else ""))

print(f"下载结束：完成 {completed}，跳过 {skipped}，失败 {len(failures)}。")
if failures:
    print("重新运行此单元可续传失败文件。")